# Aula 1 — Econometria e Causalidade

**Laboratórios de Econometria em R** · MPEF/FGV-EPGE · Prof. Marcelo Mello
Notas e código: Vítor Wilher

---

A tese do curso é que **correlação não implica causalidade**. Este notebook mostra
*com números* por que a aleatorização resolve o problema e o que exatamente se perde
sem ela.

Três resultados:

1. a comparação ingênua entre tratados e controles é grosseiramente viesada quando há
   autosseleção;
2. sob sorteio, a mesma comparação acerta o efeito causal;
3. controlar por observáveis resolve **se** o confundidor for observado — e resolve
   apenas parcialmente quando temos dele um *proxy* ruidoso.

> Rode as células em ordem (`Shift+Enter`). Tudo é simulado com semente fixa, então
> os números abaixo são reprodutíveis e não dependem de nenhum dado externo.

In [ ]:
# Pacotes. No Colab, ggplot2 e dplyr já vêm instalados; instalamos o que faltar.
for (p in c("ggplot2", "dplyr")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
}
library(ggplot2)
suppressMessages(library(dplyr))

theme_set(theme_minimal(base_size = 13))
az <- "#1f4e79"; vd <- "#2e7d32"; vm <- "#b3261e"; cz <- "grey45"

cat("R:", R.version.string, "\n")

## 1. Uma população com efeito causal conhecido

A vantagem da simulação é que **conhecemos a verdade**. Aqui o tratamento tem efeito
causal de exatamente `+5`, e existe um confundidor — chame-o *motivação* — que afeta
ao mesmo tempo a chance de receber tratamento e o resultado. É a estrutura do problema
de seleção.

In [ ]:
set.seed(2026)
n <- 20000
motivacao     <- rnorm(n)   # confundidor NÃO observado
efeito_causal <- 5          # o parâmetro que queremos recuperar

# resultados potenciais
Y0 <- 50 + 8 * motivacao + rnorm(n, sd = 5)   # sem tratamento
Y1 <- Y0 + efeito_causal                      # com tratamento

# ESCOLHA: os mais motivados se autosselecionam ao tratamento
p_trat <- plogis(1.5 * motivacao)
D_obs  <- rbinom(n, 1, p_trat)   # mundo observacional
D_ale  <- rbinom(n, 1, 0.5)      # mundo aleatorizado (sorteio)

c(efeito_verdadeiro = efeito_causal,
  prop_tratada_obs  = mean(D_obs),
  prop_tratada_ale  = mean(D_ale))

## 2. O capítulo inteiro em três linhas

Comparamos a diferença de médias nos dois mundos. Só observamos um resultado por
indivíduo — é o **problema fundamental da inferência causal**.

In [ ]:
Y_obs <- ifelse(D_obs == 1, Y1, Y0)
Y_ale <- ifelse(D_ale == 1, Y1, Y0)

dif_obs <- mean(Y_obs[D_obs == 1]) - mean(Y_obs[D_obs == 0])
dif_ale <- mean(Y_ale[D_ale == 1]) - mean(Y_ale[D_ale == 0])

round(c(verdadeiro         = efeito_causal,
        observacional      = dif_obs,
        vies_observacional = dif_obs - efeito_causal,
        aleatorizado       = dif_ale,
        vies_aleatorizado  = dif_ale - efeito_causal), 3)

A comparação **observacional** superestima grosseiramente: ela atribui ao tratamento
não só os 5 pontos que ele causa, mas também a vantagem prévia dos mais motivados. A
comparação **aleatorizada** acerta, com erro compatível com ruído amostral.

## 3. Por que funciona: balanceamento

A aleatorização **não elimina** o confundidor — ela o distribui igualmente entre os
grupos, que é tudo de que precisamos.

In [ ]:
rbind(
  observacional = c(tratados  = mean(motivacao[D_obs == 1]),
                    controles = mean(motivacao[D_obs == 0]),
                    diferenca = mean(motivacao[D_obs == 1]) - mean(motivacao[D_obs == 0])),
  aleatorizado  = c(mean(motivacao[D_ale == 1]),
                    mean(motivacao[D_ale == 0]),
                    mean(motivacao[D_ale == 1]) - mean(motivacao[D_ale == 0]))
) |> round(4)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 3.6)

rot_obs <- "observacional (autosseleção)"
rot_ale <- "aleatorizado (sorteio)"
d <- rbind(
  data.frame(motivacao, grupo = ifelse(D_obs == 1, "tratado", "controle"), desenho = rot_obs),
  data.frame(motivacao, grupo = ifelse(D_ale == 1, "tratado", "controle"), desenho = rot_ale)
)
d$desenho <- factor(d$desenho, levels = c(rot_obs, rot_ale))

ggplot(d, aes(x = motivacao, fill = grupo)) +
  geom_density(alpha = 0.45, colour = NA) +
  facet_wrap(~desenho) +
  scale_fill_manual(values = c(controle = cz, tratado = az)) +
  labs(x = "motivação (confundidor não observado)", y = "densidade",
       fill = NULL) +
  theme(legend.position = "bottom")

Sob autosseleção os grupos são **visivelmente diferentes antes de qualquer tratamento**;
sob sorteio as distribuições se sobrepõem.

## 4. O que "controlar por observáveis" resolve — e o que não resolve

A aposta da regressão é que, controlando pelos observáveis, o viés desaparece. Vale
ver isso funcionar e, em seguida, falhar.

In [ ]:
m_sem <- lm(Y_obs ~ D_obs)                 # sem controle
m_com <- lm(Y_obs ~ D_obs + motivacao)     # confundidor observado

proxy   <- motivacao + rnorm(n, sd = 1.2)  # só um indicador ruidoso dele
m_proxy <- lm(Y_obs ~ D_obs + proxy)

round(c(verdadeiro        = efeito_causal,
        sem_controle      = coef(m_sem)[2],
        com_confundidor   = coef(m_com)[2],
        com_proxy_ruidoso = coef(m_proxy)[2]), 4)

Três resultados, três lições:

| situação | resultado |
|---|---|
| sem controle | viés grande |
| confundidor observado e incluído | recupera o efeito quase exatamente |
| apenas um *proxy* ruidoso | viés **diminui, mas não desaparece** |

O terceiro caso é o realista. Raramente observamos o confundidor; em geral temos
indicadores imperfeitos dele. Daí a advertência do curso: a hipótese de identificação
é uma **aposta substantiva**, que depende de conhecimento do problema — não algo que
os dados possam confirmar sozinhos.

## Para experimentar

1. Aumente `sd = 1.2` do *proxy* para 3 e veja o viés crescer: quanto pior a medida,
   menos ela corrige.
2. Torne o efeito **heterogêneo** (por exemplo `Y1 <- Y0 + 5 + 2 * motivacao`). Sob
   sorteio, a diferença de médias ainda recupera algo interpretável — o quê?
3. Reduza `n` para 200 e rode várias vezes: separe o que é **viés** (não some com `n`)
   do que é **ruído** (some).

---

[🏠 Índice](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/00-indice.ipynb) · ➡️ [**Aula 2 — Probabilidade**](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/02-probabilidade.ipynb)